In [ ]:
# pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
# pip install transformers==4.41.1

In [2]:
from transformers import LongT5ForConditionalGeneration, AutoTokenizer
from rouge_score import rouge_scorer
import torch


text_to_summarize = """
In this paper the problem of the existence of the periodicity of about 155 days during the maximum activity period 
for sunspot data from 1923 - 1933 (cycle 16) is considered. The daily sunspot areas, the mean sunspot areas per 
Carrington rotation, the monthly sunspot numbers and their fluctuations, which are obtained after removing the 11-year 
cycle are analysed. A new method of the diagnosis of an echo-effect in the power spectrum is presented. Numerical results 
of the new method are presented.
"""

reference_summary = """The paper explores the periodicity of approximately 155 days in sunspot activity during 1923–1933, using various data and a new diagnostic method for echo effects in power spectra."""

# Load model and tokenizer
# The transformers library automatically detects and loads .safetensors files
# if they are present in the model directory.
model = LongT5ForConditionalGeneration.from_pretrained("./longt5_best_model")
tokenizer = AutoTokenizer.from_pretrained("./longt5_best_model")

# Force the device to be CPU
device = torch.device("cpu")
model.to(device)
model.eval()

# Preprocess and generate
input_text = "summarize: " + text_to_summarize
# Ensure the tensors are on the CPU
inputs = tokenizer(input_text, return_tensors="pt", max_length=4096, truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=256,
    min_length=30,
    length_penalty=2.0,
    repetition_penalty=1.2,
    num_beams=4,
    early_stopping=True
)

generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\nGenerated Summary:\n", generated_summary)

# # Evaluate with ROUGE
if reference_summary:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    score = scorer.score(reference_summary, generated_summary)

    print("\nROUGE Scores:")
    for k, v in score.items():
        print(f"{k}: {v.fmeasure:.4f}")

c:\Users\Dell\anaconda3\envs\testInfo\lib\site-packages\transformers\modeling_utils.py:1006: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(



Generated Summary:
 sunspot data from 1923 - 1933 (cycle 16) are analysed. a new method of the diagnosis of an echo-effect in the power spectrum is presented.

ROUGE Scores:
rouge1: 0.5185
rouge2: 0.1538
rougeL: 0.3704
